# 08 — AI-Ready — ALL MODELS Results & Evidence Insights (v2.1) (PyCIL)

**Tujuan:** mengubah hasil **production** EFCIL × Attention menjadi *evidence pack* untuk dianalisis secara transparan oleh AI (CSV + JSON + Markdown + ZIP). Notebook ini adalah **pelengkap**, bukan pengganti `07_results_analysis.ipynb`.

- Membaca hasil dari **PC2** dan ekspor **server RTX A5000**.
- Menggunakan **canonical predictions** (FeTrIL=SVM, FeCAM=Mahalanobis, SSRE/EFC/PASS sesuai runner) untuk audit, bukan accuracy smoke test.
- Mengeluarkan data untuk semua metode, backbone, protocol, dan attention yang tersedia.
- Memisahkan observasi numerik yang tervalidasi dari interpretasi sebab-akibat yang *harus* diperiksa kembali.
- **Tidak mengirim data ke layanan AI** dan tidak membutuhkan API key. Setelah dihasilkan, upload ZIP laporan ke ChatGPT/AI yang Anda pilih.

> Tidak menyertakan dataset satelit atau checkpoint `.pkl` di ZIP final. `ready_snapshot` berarti inventory checkpoint ada pada mesin sumber, bukan weights berhasil dimuat.


## 1 — Prasyarat & lokasi file

Simpan dua file dalam proyek yang sama:

```text
PyCIL/
├── notebooks/08_ai_ready_insights.ipynb
├── scripts/ai_ready_results_export.py
├── outputs/experiments/<run_id>/...            # produksi PC2
└── analysis_inputs/server/experiments/<run_id>/... # hasil ZIP server yang telah diekstrak
```

Ekspor server dapat dibuat dengan `python scripts/export_evaluation_artifacts.py --machine server`, lalu pindahkan dan ekstrak ke `analysis_inputs/server/` di PC2. Gunakan kernel Python yang memiliki `numpy` dan `pandas`. Ekspor hanya saat tidak ada file output training yang sedang ditulis.

In [1]:
from pathlib import Path
import importlib.util
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = CURRENT.parent if CURRENT.name == "notebooks" else CURRENT
ENGINE = PROJECT_ROOT / "scripts" / "ai_ready_results_export.py"
if not ENGINE.is_file():
    raise FileNotFoundError(f"Script tidak ditemukan: {ENGINE}. Simpan script ke PyCIL/scripts/.")
spec = importlib.util.spec_from_file_location("ai_ready_results_export", ENGINE)
ai_exporter = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ai_exporter)
EXPECTED_EXPORTER_VERSION = "2.1.0"
EXPORTER_VERSION = getattr(ai_exporter, "AI_READY_EXPORTER_VERSION", None)
if EXPORTER_VERSION != EXPECTED_EXPORTER_VERSION:
    raise RuntimeError(
        "VERSI EXPORTER TIDAK COCOK. Notebook memerlukan "
        f"ai_ready_results_export.py v{EXPECTED_EXPORTER_VERSION}, "
        f"tetapi ditemukan versi {EXPORTER_VERSION or 'lama/tidak dikenal'} di: {ENGINE}\n"
        "Ganti file PyCIL/scripts/ai_ready_results_export.py dengan versi dari paket v2.1, "
        "kemudian jalankan ulang notebook dari Cell 2 (atau Restart Kernel & Run All)."
    )
print("PyCIL root:", PROJECT_ROOT)
print("Exporter:", ENGINE, "| version:", EXPORTER_VERSION)
print("pandas:", pd.__version__, "numpy:", np.__version__)


PyCIL root: C:\Skripsi\Project\PyCIL
Exporter: C:\Skripsi\Project\PyCIL\scripts\ai_ready_results_export.py | version: 2.1.0
pandas: 2.3.3 numpy: 1.26.4


## 2 — Konfigurasi sumber hasil & cakupan

Pengaturan default mengikuti hasil AID production (80 konfigurasi), seed 1993. Ganti `DATASET_SCOPE` menjadi `"all"` **hanya jika** hasil NWPU sudah tersedia. Jika nama kelas tidak dapat dipetakan secara benar dari `splits/.../train.csv`, buat CSV manual `analysis_inputs/class_labels.csv` dengan kolom `dataset,incremental_id,class_name`. Jangan menebak urutan kelas.

In [2]:
DATASET_SCOPE = "aid"      # "aid" | "nwpu" | "all"
TRAINING_SEED = 1993
SOURCES = {
    "pc2": PROJECT_ROOT / "outputs" / "experiments",
    "server": PROJECT_ROOT / "analysis_inputs" / "server" / "experiments",
}
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analysis_ai_ready"
LABELS_FILE = PROJECT_ROOT / "analysis_inputs" / "class_labels.csv"
LABELS_FILE = LABELS_FILE if LABELS_FILE.exists() else None

for machine, path in SOURCES.items():
    print(machine, "=>", path, "| exists:", path.is_dir())
print("Labels override:", LABELS_FILE or "none; read from manifest if possible, otherwise Cxx")
print("Output directory:", OUTPUT_DIR)


pc2 => C:\Skripsi\Project\PyCIL\outputs\experiments | exists: True
server => C:\Skripsi\Project\PyCIL\analysis_inputs\server\experiments | exists: True
Labels override: none; read from manifest if possible, otherwise Cxx
Output directory: C:\Skripsi\Project\PyCIL\outputs\analysis_ai_ready


## 3 — Audit, perhitungan, insight otomatis, dan ZIP

Cell berikut memverifikasi kelengkapan run, kecocokan *confusion matrix* vs `y_true`/`y_pred`, konsistensi AIA/LTA, checkpoint manifest/inventory, serta fairness pairing sebelum menghitung delta. Hasil partial **tidak** dianggap valid.

In [3]:
if getattr(ai_exporter, "AI_READY_EXPORTER_VERSION", None) != EXPECTED_EXPORTER_VERSION:
    raise RuntimeError("Objek exporter di kernel masih versi lama. Jalankan ulang Cell 2 atau Restart Kernel & Run All.")
result = ai_exporter.main(
    project_root=PROJECT_ROOT,
    sources=SOURCES,
    output_dir=OUTPUT_DIR,
    dataset_scope=DATASET_SCOPE,
    seed=TRAINING_SEED,
    class_labels_file=LABELS_FILE,
    create_zip=True,
    allow_inventory=True,
)
if result["manifest"].get("schema_version") != EXPECTED_EXPORTER_VERSION:
    raise RuntimeError("Versi manifest tidak sesuai notebook: " + str(result["manifest"].get("schema_version")))
REQUIRED_FILES = list(ai_exporter.REQUIRED_ALL_MODELS_OUTPUTS)
MISSING = [n for n in REQUIRED_FILES if not (OUTPUT_DIR / n).is_file()]
if MISSING:
    raise RuntimeError(
        "EXPORT TIDAK LENGKAP. Missing: " + ", ".join(MISSING)
        + ". Periksa status Cell 6 dan pastikan scripts/ai_ready_results_export.py v2.1."
    )
print("VALIDASI OUTPUT ALL MODELS: PASSED")
for n in REQUIRED_FILES:
    print("  OK:", n, f"({(OUTPUT_DIR / n).stat().st_size:,} bytes)")
print(json.dumps(result["manifest"], indent=2, ensure_ascii=False))
print("\nAI-READY ZIP:", result["zip_path"])


VALIDASI OUTPUT ALL MODELS: PASSED
  OK: model_insights_all.csv (11,480 bytes)
  OK: bundle_insights_all.csv (4,505 bytes)
  OK: attention_insights_all.csv (24,670 bytes)
  OK: class_insights_final_all.csv (337,122 bytes)
  OK: insights_all.json (172,051 bytes)
  OK: AI_EVIDENCE_REPORT.md (76,672 bytes)
{
  "schema_version": "2.1.0",
  "generated_at": "2026-11-11T14:03:45+00:00",
  "dataset_scope": [
    "aid"
  ],
  "seed": 1993,
  "counts": {
    "candidate_runs": 80,
    "valid_runs": 80,
    "expected_runs": 80,
    "ready_expected": 80,
    "paired_effects": 60,
    "rejected_pairs": 0,
    "class_observations": 12800,
    "misclassification_pairs": 15850,
    "model_insights_all": 80,
    "bundle_insights_all": 20,
    "expected_attention_pairs": 60,
    "paired_insights_valid": 60,
    "class_insights_final_all": 1800
  },
  "sources": {
    "pc2": {
      "present": true
    },
    "server": {
      "present": true
    }
  },
  "ready_full_count": 0,
  "ready_snapshot_count": 8

## 4 — Status, coverage, dan daftar run yang perlu diperbaiki

`expected_coverage.csv` memiliki **80 baris untuk AID**: 5 metode × 2 backbone × 2 protocol × 4 attention. Jika belum 80/80, insight hanya bersifat sementara.

In [4]:
audit = pd.read_csv(OUTPUT_DIR / "run_audit.csv")
coverage = pd.read_csv(OUTPUT_DIR / "expected_coverage.csv")
valid_runs = pd.read_csv(OUTPUT_DIR / "valid_runs.csv")
print("Valid", int(coverage.ready.sum()), "/", len(coverage))
if not audit.empty:
    display(audit.groupby("status", dropna=False).size().rename("n_runs").reset_index())
    bad = audit[~audit.status.isin(["ready_full", "ready_snapshot", "duplicate_copy"])]
    if not bad.empty:
        display(bad[["run_id", "source_machine", "status", "issue"]].head(20))
if not valid_runs.empty:
    display(valid_runs[["dataset","method","protocol","backbone","attention","aia","lta","status","run_id"]].head(12))


Valid 80 / 80


,status,n_runs
0,ready_snapshot,80


,dataset,method,protocol,backbone,attention,aia,lta,status,run_id
0,aid,fecam,T10,r18,cbam,85.386181,79.25,ready_snapshot,aid_fecam_t10_r18_cbam_seed1993
1,aid,fecam,T10,r18,eca,83.411589,76.10,ready_snapshot,aid_fecam_t10_r18_eca_seed1993
2,aid,fecam,T10,r18,none,83.161841,76.20,ready_snapshot,aid_fecam_t10_r18_none_seed1993
3,aid,fecam,T10,r18,procbam,85.825244,79.90,ready_snapshot,aid_fecam_t10_r18_procbam_seed1993
4,aid,fecam,T5,r18,cbam,88.366545,83.10,ready_snapshot,aid_fecam_t5_r18_cbam_seed1993
5,aid,fecam,T5,r18,eca,86.943795,80.80,ready_snapshot,aid_fecam_t5_r18_eca_seed1993
6,aid,fecam,T5,r18,none,87.658371,81.45,ready_snapshot,aid_fecam_t5_r18_none_seed1993
7,aid,fecam,T5,r18,procbam,88.323391,82.45,ready_snapshot,aid_fecam_t5_r18_procbam_seed1993
8,aid,pass,T10,r18,cbam,64.436470,39.35,ready_snapshot,aid_pass_t10_r18_cbam_seed1993
9,aid,pass,T10,r18,eca,74.595409,58.85,ready_snapshot,aid_pass_t10_r18_eca_seed1993


## 5 — Attention Gain dan fairness check

- **ΔAIA**: AIA dengan attention − AIA baseline None (pp).
- **Mean ΔAIA**: rata-rata delta dari 4 pasangan `R18/R34 × T5/T10` per metode **jika lengkap**.
- `n_pairs=4` bukan empat seed independen. Pasangan yang machine/batch/histogram evaluasinya berbeda akan ditolak.

In [5]:
paired = pd.read_csv(OUTPUT_DIR / "paired_attention_effects.csv")
gain = pd.read_csv(OUTPUT_DIR / "method_attention_gain.csv")
rejected = pd.read_csv(OUTPUT_DIR / "rejected_pairs.csv")
if not gain.empty:
    display(gain[["method","attention","n_pairs","expected_pairs","complete_bundle","mean_delta_aia_pp","mean_delta_lta_pp","win_rate_pct"]].sort_values("mean_delta_aia_pp",ascending=False).round(3))
if not rejected.empty:
    display(Markdown("**Pasangan yang ditolak untuk perbandingan attention:**"))
    display(rejected)
if not paired.empty:
    display(paired[["method","protocol","backbone","attention","delta_aia_pp","delta_lta_pp","baseline_run_id","attention_run_id"]].head(20).round(3))


,method,attention,n_pairs,expected_pairs,complete_bundle,mean_delta_aia_pp,mean_delta_lta_pp,win_rate_pct
0,efc,procbam,4,4,True,5.429,6.512,100.0
1,efc,cbam,4,4,True,4.864,5.425,100.0
2,ssre,eca,4,4,True,4.472,6.250,100.0
3,pass,eca,4,4,True,4.436,6.113,100.0
4,ssre,procbam,4,4,True,3.997,4.300,100.0
5,fetril,eca,4,4,True,3.520,5.163,100.0
6,ssre,cbam,4,4,True,2.829,2.862,100.0
7,pass,procbam,4,4,True,2.706,1.737,100.0
8,fetril,cbam,4,4,True,2.260,3.125,100.0
9,fetril,procbam,4,4,True,1.817,2.275,100.0


,method,protocol,backbone,attention,delta_aia_pp,delta_lta_pp,baseline_run_id,attention_run_id
0,efc,T10,r18,cbam,6.763,7.10,aid_efc_t10_r18_none_seed1993,aid_efc_t10_r18_cbam_seed1993
1,efc,T10,r18,eca,0.784,1.50,aid_efc_t10_r18_none_seed1993,aid_efc_t10_r18_eca_seed1993
2,efc,T10,r18,procbam,6.185,8.25,aid_efc_t10_r18_none_seed1993,aid_efc_t10_r18_procbam_seed1993
3,efc,T10,r34,cbam,4.426,4.95,aid_efc_t10_r34_none_seed1993,aid_efc_t10_r34_cbam_seed1993
4,efc,T10,r34,eca,0.637,1.20,aid_efc_t10_r34_none_seed1993,aid_efc_t10_r34_eca_seed1993
5,efc,T10,r34,procbam,5.945,6.10,aid_efc_t10_r34_none_seed1993,aid_efc_t10_r34_procbam_seed1993
6,efc,T5,r18,cbam,4.571,5.55,aid_efc_t5_r18_none_seed1993,aid_efc_t5_r18_cbam_seed1993
7,efc,T5,r18,eca,1.478,2.40,aid_efc_t5_r18_none_seed1993,aid_efc_t5_r18_eca_seed1993
8,efc,T5,r18,procbam,5.515,7.15,aid_efc_t5_r18_none_seed1993,aid_efc_t5_r18_procbam_seed1993
9,efc,T5,r34,cbam,3.696,4.10,aid_efc_t5_r34_none_seed1993,aid_efc_t5_r34_cbam_seed1993


## 6 — Class accuracy: semua attention & tiap task

`class_accuracy_pct` di sini adalah **recall per kelas** (`TP / jumlah sampel sebenarnya dari kelas tersebut`). Bandingkan keempat attention setelah mengecek `class_label_verified` dan jumlah sampel `support`. Kelas yang belum diketahui namanya tetap ditampilkan sebagai `C00`, `C01`, dan seterusnya.

In [6]:
CLASS_METHOD = "pass"     # fetril | fecam | ssre | efc | pass
CLASS_PROTOCOL = "T5"
CLASS_BACKBONE = "r18"

wide = pd.read_csv(OUTPUT_DIR / "class_attention_final_wide.csv")
class_long = pd.read_csv(OUTPUT_DIR / "class_accuracy_long.csv")
class_delta = pd.read_csv(OUTPUT_DIR / "class_attention_delta.csv")
selection = wide[(wide.method == CLASS_METHOD)&(wide.protocol == CLASS_PROTOCOL)&
                 (wide.backbone == CLASS_BACKBONE)] if not wide.empty else wide
if selection.empty:
    print("Model yang dipilih belum memiliki hasil class accuracy.")
else:
    cols=[c for c in ["class_id","class_name","support","none","eca","cbam","procbam","delta_eca_pp","delta_cbam_pp","delta_procbam_pp"] if c in selection]
    display(selection[cols].round(2).head(45))
    if not class_delta.empty:
        tmp=class_delta[(class_delta.method==CLASS_METHOD)&(class_delta.protocol==CLASS_PROTOCOL)&
                        (class_delta.backbone==CLASS_BACKBONE)].copy()
        if not tmp.empty:
            display(tmp[["task_id","class_id","class_name","attention","baseline_class_accuracy_pct","attention_class_accuracy_pct","delta_class_accuracy_pp","attention_support"]].sort_values("delta_class_accuracy_pp").head(10).round(2))


,class_id,class_name,support,none,eca,cbam,procbam,delta_eca_pp,delta_cbam_pp,delta_procbam_pp
420,0,Stadium,58,39.66,68.97,39.66,50.00,29.31,0.00,10.34
421,1,RailwayStation,52,36.54,48.08,53.85,71.15,11.54,17.31,34.62
422,2,Square,66,45.45,57.58,51.52,62.12,12.12,6.06,16.67
423,3,BaseballField,44,40.91,59.09,45.45,59.09,18.18,4.55,18.18
424,4,DenseResidential,82,64.63,76.83,75.61,82.93,12.20,10.98,18.29
425,5,Meadow,56,57.14,75.00,73.21,91.07,17.86,16.07,33.93
426,6,Farmland,74,72.97,86.49,74.32,86.49,13.51,1.35,13.51
427,7,Church,48,72.92,85.42,81.25,81.25,12.50,8.33,8.33
428,8,Mountain,68,60.29,60.29,55.88,58.82,0.00,-4.41,-1.47
429,9,Port,76,47.37,78.95,69.74,68.42,31.58,22.37,21.05


,task_id,class_id,class_name,attention,baseline_class_accuracy_pct,attention_class_accuracy_pct,delta_class_accuracy_pp,attention_support
1681,4,11,River,cbam,64.63,43.90,-20.73,82
1850,2,8,Mountain,procbam,85.29,70.59,-14.71,68
1695,4,25,Beach,cbam,73.75,60.00,-13.75,80
1627,2,5,Meadow,cbam,94.64,82.14,-12.50,56
1625,2,3,BaseballField,cbam,75.00,65.91,-9.09,44
1647,3,3,BaseballField,cbam,50.00,40.91,-9.09,44
1759,3,5,Meadow,eca,83.93,75.00,-8.93,56
1869,3,5,Meadow,procbam,83.93,75.00,-8.93,56
1824,1,0,Stadium,procbam,72.41,63.79,-8.62,58
1915,4,25,Beach,procbam,73.75,66.25,-7.50,80


## 7 — Per-class forgetting, old/new accuracy, dan perubahan confusion matrix

Dua pertanyaan berbeda: apakah accuracy meningkat pada kelas baru (*plasticity*) dan apakah akurasi kelas lama bertahan (*stability*)? Jangan menyimpulkan attention lebih baik hanya dari satu kelas. Tinjau `task_attention_delta.csv`, `class_forgetting_attention_delta.csv`, dan `misclassification_delta.csv`.

In [7]:
task_delta = pd.read_csv(OUTPUT_DIR / "task_attention_delta.csv")
forget_delta = pd.read_csv(OUTPUT_DIR / "class_forgetting_attention_delta.csv")
mis_delta = pd.read_csv(OUTPUT_DIR / "misclassification_delta.csv")
if not task_delta.empty:
    print("Task-level old/new accuracy delta:")
    display(task_delta[["method","protocol","backbone","attention","task_id","delta_top1_pp","delta_old_pp","delta_new_pp"]].head(20).round(2))
if not forget_delta.empty:
    print("Per-class forgetting delta (negative = less forgetting):")
    display(forget_delta.sort_values("delta_forgetting_pp").head(12).round(2))
if not mis_delta.empty:
    print("Misclassification directions most reduced:")
    display(mis_delta.sort_values("errors_reduced",ascending=False).head(12)[["method","protocol","backbone","attention","true_class","predicted_class","support","errors_none","errors_attention","errors_reduced"]])


Task-level old/new accuracy delta:


,method,protocol,backbone,attention,task_id,delta_top1_pp,delta_old_pp,delta_new_pp
0,efc,T10,r18,cbam,0,0.00,NaN,0.00
1,efc,T10,r18,cbam,1,9.85,11.78,0.00
2,efc,T10,r18,cbam,2,7.68,8.53,0.00
3,efc,T10,r18,cbam,3,6.02,5.61,9.23
4,efc,T10,r18,cbam,4,8.67,9.42,2.74
5,efc,T10,r18,cbam,5,6.82,7.51,0.69
6,efc,T10,r18,cbam,6,6.77,6.55,9.23
7,efc,T10,r18,cbam,7,7.11,7.98,-2.00
8,efc,T10,r18,cbam,8,7.61,8.33,0.00
9,efc,T10,r18,cbam,9,7.10,7.77,-3.28


Per-class forgetting delta (negative = less forgetting):


,dataset,method,protocol,backbone,seed,attention,class_id,class_name,baseline_run_id,attention_run_id,baseline_forgetting_pp,attention_forgetting_pp,delta_forgetting_pp,support
1599,aid,ssre,T10,r34,1993,procbam,9,Port,aid_ssre_t10_r34_none_seed1993,aid_ssre_t10_r34_procbam_seed1993,82.89,26.32,-56.58,76
1569,aid,ssre,T10,r34,1993,eca,9,Port,aid_ssre_t10_r34_none_seed1993,aid_ssre_t10_r34_eca_seed1993,82.89,26.32,-56.58,76
1470,aid,ssre,T10,r18,1993,eca,0,Stadium,aid_ssre_t10_r18_none_seed1993,aid_ssre_t10_r18_eca_seed1993,74.14,24.14,-50.00,58
1539,aid,ssre,T10,r34,1993,cbam,9,Port,aid_ssre_t10_r34_none_seed1993,aid_ssre_t10_r34_cbam_seed1993,82.89,32.89,-50.00,76
3,aid,efc,T10,r18,1993,cbam,3,BaseballField,aid_efc_t10_r18_none_seed1993,aid_efc_t10_r18_cbam_seed1993,59.09,13.64,-45.45,44
1114,aid,pass,T10,r18,1993,eca,4,DenseResidential,aid_pass_t10_r18_none_seed1993,aid_pass_t10_r18_eca_seed1993,73.17,30.49,-42.68,82
862,aid,fetril,T10,r34,1993,eca,22,Desert,aid_fetril_t10_r34_none_seed1993,aid_fetril_t10_r34_eca_seed1993,55.00,13.33,-41.67,60
1487,aid,ssre,T10,r18,1993,eca,17,Resort,aid_ssre_t10_r18_none_seed1993,aid_ssre_t10_r18_eca_seed1993,44.83,5.17,-39.66,58
63,aid,efc,T10,r18,1993,procbam,3,BaseballField,aid_efc_t10_r18_none_seed1993,aid_efc_t10_r18_procbam_seed1993,59.09,20.45,-38.64,44
61,aid,efc,T10,r18,1993,procbam,1,RailwayStation,aid_efc_t10_r18_none_seed1993,aid_efc_t10_r18_procbam_seed1993,78.85,40.38,-38.46,52


Misclassification directions most reduced:


,method,protocol,backbone,attention,true_class,predicted_class,support,errors_none,errors_attention,errors_reduced
2619,fecam,T10,r18,cbam,Desert,BareLand,60,34,8,26
2892,fecam,T10,r18,eca,Desert,BareLand,60,34,8,26
702,efc,T10,r34,cbam,Port,Parking,76,31,6,25
7026,fetril,T10,r34,eca,Desert,BareLand,60,36,11,25
13796,ssre,T10,r34,procbam,Port,Bridge,76,31,7,24
60,efc,T10,r18,cbam,Port,Parking,76,35,11,24
480,efc,T10,r18,procbam,Port,Parking,76,35,11,24
15492,ssre,T5,r34,eca,Desert,BareLand,60,41,17,24
9264,pass,T10,r18,eca,DenseResidential,School,82,27,4,23
7334,fetril,T10,r34,procbam,Desert,BareLand,60,36,13,23


## 8 — Highlight otomatis dan laporan lengkap seluruh model yang dapat diaudit

Insight dibuat dengan aturan deterministik (tanpa LLM). Setiap insight menyertakan **angka dan file bukti**. Anda atau AI tetap perlu mengecek konsistensi, keterbatasan sampel, dan mekanisme ilmiahnya.
**Penting:** `insights.json` adalah highlight terpilih, bukan cakupan seluruh model. Analisis lengkap ada pada `insights_all.json`, `model_insights_all.csv`, `bundle_insights_all.csv`, `attention_insights_all.csv`, dan `class_insights_final_all.csv`.


In [8]:
insights_doc = json.loads((OUTPUT_DIR / "insights.json").read_text(encoding="utf-8"))
print("Jumlah temuan:", insights_doc["count"])
for item in insights_doc["insights"][:12]:
    evidence = ", ".join(e["file"] for e in item["evidence"])
    display(Markdown(f"**{item['insight_id']} — {item['title']}**\n\n{item['claim']}\n\n*Bukti: {evidence}*"))
print("Seluruh temuan ada di insights.json dan AI_EVIDENCE_REPORT.md")


Jumlah temuan: 30


**I001 — Kelengkapan hasil production**

80 dari 80 konfigurasi pada grid yang dipilih lolos audit artefak.

*Bukti: expected_coverage.csv*

**I002 — AIA tertinggi yang teramati untuk T10**

FECAM R18 PROCBAM pada T10 memiliki AIA 85.83% dari hasil valid yang tersedia.

*Bukti: valid_runs.csv*

**I003 — AIA tertinggi yang teramati untuk T5**

FECAM R18 CBAM pada T5 memiliki AIA 88.37% dari hasil valid yang tersedia.

*Bukti: valid_runs.csv*

**I004 — Metode paling responsif terhadap ECA pada AID (bundle lengkap)**

SSRE mempunyai Mean ΔAIA +4.47 pp dari 4 pasangan sebanding.

*Bukti: method_attention_gain.csv*

**I005 — Metode paling responsif terhadap CBAM pada AID (bundle lengkap)**

EFC mempunyai Mean ΔAIA +4.86 pp dari 4 pasangan sebanding.

*Bukti: method_attention_gain.csv*

... (4 outputs left out)

**I010 — Peningkatan akurasi kelas di task final**

SSRE R34 T10 / ECA, kelas Port: 14.47% menjadi 73.68% (+59.21 pp; n=76).

*Bukti: class_attention_delta.csv*

**I011 — Peningkatan akurasi kelas di task final**

SSRE R34 T10 / PROCBAM, kelas Port: 14.47% menjadi 72.37% (+57.89 pp; n=76).

*Bukti: class_attention_delta.csv*

**I012 — Peningkatan akurasi kelas di task final**

FECAM R34 T10 / CBAM, kelas Desert: 26.67% menjadi 83.33% (+56.67 pp; n=60).

*Bukti: class_attention_delta.csv*

Seluruh temuan ada di insights.json dan AI_EVIDENCE_REPORT.md


## 9 — Verifikasi cakupan: SEMUA 80 model, 20 bundle, dan 60 pasangan AID

Bagian ini **wajib** untuk memastikan tidak ada konfigurasi yang hanya terwakili oleh contoh *top-k*. File `model_insights_all.csv` mencakup baseline None juga. File `attention_insights_all.csv` mencakup semua pasangan dengan status `paired_valid` atau alasan belum tersedia. Jika hasil production AID selesai 80/80, harus ada **80 model, 20 bundle, 60 pasangan valid**, dan **1.800 rekaman kelas final** (60 pasangan × 30 kelas). Tidak ada analisis yang diisi dengan angka palsu ketika output belum selesai.


In [9]:
all_models = pd.read_csv(OUTPUT_DIR / "model_insights_all.csv")
all_bundles = pd.read_csv(OUTPUT_DIR / "bundle_insights_all.csv")
all_pairs = pd.read_csv(OUTPUT_DIR / "attention_insights_all.csv")
all_classes_final = pd.read_csv(OUTPUT_DIR / "class_insights_final_all.csv")

expected_models = 80 if DATASET_SCOPE == "aid" else (160 if DATASET_SCOPE == "all" else 80)
expected_pairs = expected_models // 4 * 3
print(f"Expected model configurations: {expected_models} | Exported: {len(all_models)}")
print(f"Expected comparison bundles: {expected_models//4} | Exported: {len(all_bundles)}")
print(f"Expected attention pairs: {expected_pairs} | Exported: {len(all_pairs)}")
print(f"Fair pairs: {(all_pairs.status == 'paired_valid').sum()} | Other statuses: {(all_pairs.status != 'paired_valid').sum()}")
print(f"Final per-class comparisons: {len(all_classes_final)}")
assert len(all_models) == expected_models, "Some model configurations were omitted"
assert len(all_bundles) == expected_models // 4, "Some bundles were omitted"
assert len(all_pairs) == expected_pairs, "Some attention comparisons were omitted"
if all_models.ready.all():
    assert all_pairs.status.eq('paired_valid').all(), "All models ready but some comparisons failed fairness"
    expected_class_rows = 60*30 if DATASET_SCOPE == 'aid' else (60*45 if DATASET_SCOPE == 'nwpu' else 60*(30+45))
    assert len(all_classes_final) == expected_class_rows, "Some final per-class comparisons were omitted"
    print("FULL COVERAGE VALIDATION: PASSED")
else:
    print("Coverage partial; missing model/pair rows remain explicitly flagged")

print("\nFull model summary (no .head() cutoff):")
with pd.option_context('display.max_rows', 200, 'display.max_columns', 14):
    display(all_models[['dataset','method','protocol','backbone','attention','status','aia_pct','lta_pct','macro_f1_pct','run_id']])


Expected model configurations: 80 | Exported: 80
Expected comparison bundles: 20 | Exported: 20
Expected attention pairs: 60 | Exported: 60
Fair pairs: 60 | Other statuses: 0
Final per-class comparisons: 1800
FULL COVERAGE VALIDATION: PASSED

Full model summary (no .head() cutoff):


,dataset,method,protocol,backbone,attention,status,aia_pct,lta_pct,macro_f1_pct,run_id
0,aid,fetril,T5,r18,none,ready_snapshot,85.1122,77.80,77.7300,aid_fetril_t5_r18_none_seed1993
1,aid,fetril,T5,r18,eca,ready_snapshot,87.8103,81.45,81.3639,aid_fetril_t5_r18_eca_seed1993
2,aid,fetril,T5,r18,cbam,ready_snapshot,86.5015,78.75,78.7127,aid_fetril_t5_r18_cbam_seed1993
3,aid,fetril,T5,r18,procbam,ready_snapshot,85.6727,77.50,77.2414,aid_fetril_t5_r18_procbam_seed1993
4,aid,fetril,T5,r34,none,ready_snapshot,84.4092,76.05,75.9757,aid_fetril_t5_r34_none_seed1993
5,aid,fetril,T5,r34,eca,ready_snapshot,86.1343,78.30,78.4082,aid_fetril_t5_r34_eca_seed1993
6,aid,fetril,T5,r34,cbam,ready_snapshot,87.6291,80.70,80.4473,aid_fetril_t5_r34_cbam_seed1993
7,aid,fetril,T5,r34,procbam,ready_snapshot,85.8807,78.50,78.3830,aid_fetril_t5_r34_procbam_seed1993
8,aid,fetril,T10,r18,none,ready_snapshot,78.2576,67.25,67.8079,aid_fetril_t10_r18_none_seed1993
9,aid,fetril,T10,r18,eca,ready_snapshot,82.6092,74.40,74.1364,aid_fetril_t10_r18_eca_seed1993


## 10 — Semua perbandingan per-model + insight per kelas

Tabel berikut menampilkan **seluruh 60 perbandingan AID** (atau 120 ketika kedua dataset dipilih). Masing-masing baris menjelaskan ΔAIA/LTA dan jumlah kelas yang meningkat/menurun/tetap, bukan sekadar lima kasus terbesar. Untuk angka setiap kelas dan task, AI harus membaca `class_insights_final_all.csv`, `class_attention_delta.csv`, dan `class_accuracy_long.csv`.


In [10]:
cols = [
    'dataset','method','protocol','backbone','attention','status',
    'delta_aia_pp','delta_lta_pp','delta_macro_f1_pp','delta_fr_pp',
    'classes_improved','classes_degraded','classes_unchanged',
    'best_class_name','best_class_delta_pp',
    'worst_class_name','worst_class_delta_pp',
]
cols = [c for c in cols if c in all_pairs]
with pd.option_context('display.max_rows', 200, 'display.max_columns', 22):
    display(all_pairs[cols].round(3))

print("\nCoverage per metode dan backbone:")
with pd.option_context('display.max_rows', 60):
    display(all_pairs.groupby(['dataset','method','backbone','protocol','status']).size().rename('pairs').reset_index())

print("\nClass accuracy detail (file contains all class rows):")
print("class_insights_final_all.csv rows:",len(all_classes_final))
if not all_classes_final.empty:
    display(all_classes_final[['method','protocol','backbone','attention','class_id','class_name',
                               'baseline_class_accuracy_pct','attention_class_accuracy_pct',
                               'delta_class_accuracy_pp']].head(12).round(2))
print("\nFull report:", OUTPUT_DIR / 'AI_EVIDENCE_REPORT.md')
print("Full JSON:", OUTPUT_DIR / 'insights_all.json')


,dataset,method,protocol,backbone,attention,status,delta_aia_pp,delta_lta_pp,delta_macro_f1_pp,delta_fr_pp,classes_improved,classes_degraded,classes_unchanged,best_class_name,best_class_delta_pp,worst_class_name,worst_class_delta_pp
0,aid,efc,T10,r18,cbam,paired_valid,6.763,7.10,8.660,-6.393,21,5,4,BaseballField,40.909,Desert,-6.667
1,aid,efc,T10,r18,eca,paired_valid,0.784,1.50,3.048,-1.290,13,11,6,RailwayStation,21.154,Farmland,-25.676
2,aid,efc,T10,r18,procbam,paired_valid,6.185,8.25,9.785,-7.734,23,6,1,BaseballField,36.364,Park,-7.143
3,aid,efc,T10,r34,cbam,paired_valid,4.426,4.95,6.604,-3.847,18,10,2,Stadium,31.034,Resort,-13.793
4,aid,efc,T10,r34,eca,paired_valid,0.637,1.20,2.084,-0.488,17,11,2,Stadium,27.586,Resort,-22.414
5,aid,efc,T10,r34,procbam,paired_valid,5.945,6.10,7.363,-6.261,19,9,2,Stadium,31.034,StorageTanks,-15.278
6,aid,efc,T5,r18,cbam,paired_valid,4.571,5.55,5.406,-3.502,25,3,2,StorageTanks,18.056,Desert,-21.667
7,aid,efc,T5,r18,eca,paired_valid,1.478,2.40,2.183,-1.039,18,9,3,Stadium,15.517,Desert,-40.000
8,aid,efc,T5,r18,procbam,paired_valid,5.515,7.15,7.561,-6.414,25,2,3,Stadium,27.586,BareLand,-6.452
9,aid,efc,T5,r34,cbam,paired_valid,3.696,4.10,3.988,-2.519,22,7,1,Meadow,14.286,Desert,-35.000



Coverage per metode dan backbone:


,dataset,method,backbone,protocol,status,pairs
0,aid,efc,r18,T10,paired_valid,3
1,aid,efc,r18,T5,paired_valid,3
2,aid,efc,r34,T10,paired_valid,3
3,aid,efc,r34,T5,paired_valid,3
4,aid,fecam,r18,T10,paired_valid,3
5,aid,fecam,r18,T5,paired_valid,3
6,aid,fecam,r34,T10,paired_valid,3
7,aid,fecam,r34,T5,paired_valid,3
8,aid,fetril,r18,T10,paired_valid,3
9,aid,fetril,r18,T5,paired_valid,3



Class accuracy detail (file contains all class rows):
class_insights_final_all.csv rows: 1800


,method,protocol,backbone,attention,class_id,class_name,baseline_class_accuracy_pct,attention_class_accuracy_pct,delta_class_accuracy_pp
0,efc,T10,r18,cbam,0,Stadium,34.48,43.10,8.62
1,efc,T10,r18,cbam,1,RailwayStation,19.23,30.77,11.54
2,efc,T10,r18,cbam,2,Square,12.12,36.36,24.24
3,efc,T10,r18,cbam,3,BaseballField,40.91,81.82,40.91
4,efc,T10,r18,cbam,4,DenseResidential,50.00,57.32,7.32
5,efc,T10,r18,cbam,5,Meadow,39.29,50.00,10.71
6,efc,T10,r18,cbam,6,Farmland,72.97,82.43,9.46
7,efc,T10,r18,cbam,7,Church,54.17,68.75,14.58
8,efc,T10,r18,cbam,8,Mountain,73.53,73.53,0.00
9,efc,T10,r18,cbam,9,Port,19.74,34.21,14.47



Full report: C:\Skripsi\Project\PyCIL\outputs\analysis_ai_ready\AI_EVIDENCE_REPORT.md
Full JSON: C:\Skripsi\Project\PyCIL\outputs\analysis_ai_ready\insights_all.json


## 11 — Unggah evidence pack ke AI

File yang siap dibaca AI adalah **`outputs/analysis_ai_ready/ai_ready_evaluation_bundle.zip`**. Bundle berisi CSV per run/per class/per task, statistik attention, report Markdown, insight JSON, kamus data, dan prompt analisis.

Saat ingin menganalisis Bab IV, unggah ZIP ke ChatGPT/AI bersama prompt yang otomatis disimpan pada `AI_PROMPT.md`. File ini **tidak** menyertakan citra dataset, raw individual predictions, atau model weights. Hal itu menjaga ukuran relatif kecil dan memudahkan AI fokus pada evidence terstruktur.

**Penting:** jika hasil PC/server belum semuanya diekspor, output final akan bersifat *preliminary*. Jalankan ulang setelah semua production, recovery, dan ekspor server selesai.

In [ ]:
from zipfile import ZipFile
bundle = Path(result["zip_path"])
print("ZIP:",bundle)
print("Size (MB):",round(bundle.stat().st_size/1024**2,2))
with ZipFile(bundle) as archive:
    print("Files:",len(archive.namelist()))
    print("CRC check:","PASSED" if archive.testzip() is None else "FAILED")
print("\nSuggested prompt: ")
print((OUTPUT_DIR / "AI_PROMPT.md").read_text(encoding="utf-8")[:1100])


ZIP: C:\Skripsi\Project\PyCIL\outputs\analysis_ai_ready\ai_ready_evaluation_bundle.zip
Size (MB): 0.68
Files: 27
CRC check: PASSED

Suggested prompt: 
# Prompt untuk AI — analisis Bab IV skripsi

Saya sedang meneliti integrasi ECA, CBAM, dan ProCBAM pada lima metode EFCIL
(FeTrIL, FeCAM, SSRE, EFC, PASS), dengan ResNet-18/34 dan protocol T5/T10,
untuk scene classification citra satelit. File yang saya upload adalah keluaran
notebook `08_ai_ready_insights.ipynb` dari hasil production terverifikasi.

Tolong:
1. Baca `manifest.json`, `data_dictionary.json`, `insights.json`, `run_audit.csv`,
   `expected_coverage.csv`, dan semua CSV yang relevan. Sebutkan kelengkapan data.
2. Jawab apakah setiap attention meningkatkan AIA dan LTA dibanding None pada
   pasangan method/dataset/protocol/backbone/seed/batch/machine yang sah.
3. WAJIB bahas seluruh 80 konfigurasi dan seluruh 60 pasangan AID, bukan hanya highlight.
   Baca model_insights_all.csv, attention_insights_all.csv, bundle_insights_all.

: 